In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-3: RLAIF and Constitutional AI

**RL for LLMs, session 3.**


## What this session teaches

Comparisons by people are slow and costly. **RLAIF** (RL from AI feedback) keeps the three stages of RLHF and replaces the person by an AI **labeler**: a language model that judges answers by a written list of principles, called a **constitution**. <!-- long-ok -->

The labeler can help at each stage:

- it can pick the better of two answers, giving comparisons for the reward model
- it can criticize an answer and rewrite it, giving demonstrations for SFT of the model being trained
- it can score an answer directly, in place of a reward model

**Constitutional AI** is a recipe that uses the first two: rewritten answers for SFT, then AI comparisons for the reward model, then RL as in RLHF.

The cost moves. Fewer people are needed, but the labels carry the labeler's errors and biases, and its reading of the principles. The model being trained can exploit those, just as it can exploit a reward model trained on people.

The question to carry: what changes when a model gives the grades, and what stays the same?

By the end you can

- say where an AI labeler plugs into the stages of RLHF
- say what Constitutional AI does at each stage
- say what AI labels save and what they risk

**Where this session fits: the topics of the series, in order**

<img src="images/rllm3v2_strip.png" width=88%>

- this session starts topic 3: a model, not a person, grades


## Setting and words

**Why this slide?** The labeler vocabulary is new here; each word is said by what it does.

| word | what it means here |
|:--|:--|
| **answer** (also called a **completion**) | the text the model writes for a prompt |
| **the model being trained** | the **policy** of RLLM-2: the model whose answers the labeler judges and whose SFT data it writes |
| **labeler** | the AI model in the role the person had; it judges answers and is not trained here (our interpretation) |
| **constitution** | a list of written principles that the labeler reads when it judges |
| **soft label** | a probability that answer A is the better one, kept as a probability instead of a hard winner |
| **stage** | SFT, reward model or RL, always by name (RLLM-2) |


## RLAIF on one page: three cases

**Why this section?** RLLM-2's pipeline needed people. A model can take over in three different ways; the next sections take them in turn, with this bar to show where we are.

> **RLAIF keeps RLHF's three stages. An AI labeler, guided by a constitution, takes over from a person or from the reward model, in three cases.**

<img src="images/rllm3v2_variants.png" width=52%>

- **Case 1, RLAIF by picking:** the labeler replaces the person who compares
    - a reward model is still trained
- **Case 2, direct RLAIF (scoring):** the labeler's score replaces the reward model
- **Case 3, Constitutional AI:** the labeler rewrites answers into demonstrations, then picks as in case 1
- the labeler reads the constitution; the model being trained does not


## Case 1, RLAIF by picking: comparisons for the reward model

<img src="images/rllm3v2_bar_1.png" width=50%>

**You are here:** case 1 of 3, RLAIF by picking

**Why this section?** This is the person's job done by a model; it is what RLAIF means.

> **The labeler picks the better of two answers by a principle, and a reward model learns from its picks.**

<img src="images/rllm3v2_mcq.png" width=48%>

- the labeler is asked a multiple-choice question: the principle, two answers, "The answer is: ("
- its probabilities for "A" and "B", normalized to add to 1, form a **soft label**
- the pick gives $(\x, \y^+, \y^-)$, so the reward model trains as in RLLM-2 and RL follows as in RLHF

**Why a soft label?**

> **Point: a close call should teach less than a clear one.**

- the soft label $p$ is the labeler's probability that $\y^+$ is better: 0.9 is a clear call, 0.51 a toss-up
- a hard winner counts both the same, so toss-ups would add noise to the reward model
- the loss is RLLM-2's with $p$ in place of a hard winner: $-p\log\sigma(d)-(1-p)\log(1-\sigma(d))$, where $d$ is the reward model's gap
- so a near-tie pushes the reward model only weakly


## Case 2, direct RLAIF: the labeler's score is the reward

<img src="images/rllm3v2_bar_2.png" width=50%>

**You are here:** case 2 of 3, direct RLAIF

**Why this section?** Picking still trains a reward model, a second network. The labeler could supply the reward during RL itself.

> **Scoring (direct RLAIF, [Lee et al. 2023](https://arxiv.org/abs/2309.00267)): the labeler's score of each sampled answer is its reward, so there is no reward model.**

- it needs a score for one answer, not an A or B pick
- the labeler rates the answer from 1 to 10, averaged by the probability of each rating token
- the paper reports results at least as good as training a reward model first
- the cost: every RL sample needs a labeler call (our interpretation)
- unlike RLVR (RLLM-4), where a program checks the answer, here the score is a model's opinion


## Case 3, Constitutional AI: rewrite first

<img src="images/rllm3v2_bar_3.png" width=50%>

**You are here:** case 3 of 3, Constitutional AI

**Why this section?** Picking needs a model that already answers harmful prompts acceptably. Rewriting makes demonstrations for it without people writing them.

> **Given one answer and one principle from the constitution, the labeler critiques the answer and rewrites it. The rewrite is an SFT example for the model being trained.**

<img src="images/rllm3_revise.png" width=38%>

- picking takes a principle and **two** answers and returns a choice
- rewriting takes a principle and **one** answer and returns a critique, then a better answer
- the principle is given in the request, sampled from the constitution
- the labeler is not asked to invent one

**What rewriting produces, and why it can work**

- the SFT example is $(\x, \y')$: the prompt and the revised answer
- the first answer is discarded
- no comparison is made here; comparisons come from picking
- in the paper the same model writes the answer, critiques it and rewrites it
- why it can work (our interpretation): the model reads its own text against a principle now in front of it
- judging a given answer is often easier than writing one at once
- a blind spot of the model stays blind when it critiques itself

**Then pick, then RL: Constitutional AI stage by stage**

**Why this slide?** After the rewrites, the recipe is case 1. Each stage consumes something different from the labeler.

| stage | what the labeler supplies | which model it trains |
|:--|:--|:--|
| SFT | rewritten answers | the model being trained |
| reward model | AI picks, as soft labels (case 1) | the reward model |
| RL | rewards from that reward model | the model being trained (the policy) |

- the model fine-tuned on the rewrites writes the answers that the labeler then picks between
- the constitution was used for harmlessness only
- the helpfulness labels were human ([Bai et al. 2022](https://arxiv.org/abs/2212.08073))


## What each case gains, and what it risks

<img src="images/rllm3v2_bar_4.png" width=50%>

**You are here:** what each case costs

**Why this section?** The cost moves. This is the table to carry away; the last column says which section covers each row.

| case | who compares or scores | cost | can miss (our interpretation) | covered in |
|:--|:--|:--|:--|:--|
| RLHF (RLLM-2) | people | human time for every label | raters disagree; a learned reward model can be pushed too hard | RLLM-2 |
| 1 RLAIF by picking | a labeler | model calls | the labeler's errors and biases; still a reward model | Case 1 |
| 2 direct RLAIF | a labeler | a labeler call per RL sample | one labeler's opinion becomes the reward | Case 2 |
| 3 Constitutional AI | a labeler, by principle | model calls and writing principles | a principle read too literally or too loosely, plus case 1's risks | Case 3 |

**The cost moves; the form stays**

> **AI labels save human time, but a proxy is a proxy: the model being trained can exploit a labeler's flaws as it exploits a reward model's.**

- the labeler's biases carry into the labels: position bias (favoring A or B, so swap and average) and self-preference (favoring its own style)
- picking trains a reward model, so it can be over-optimized (RLLM-2)
- evidence: [Lee et al. 2023](https://arxiv.org/abs/2309.00267) report RLAIF comparable to RLHF on summarization and dialogue
- [Bai et al. 2022](https://arxiv.org/abs/2212.08073) report answers that are harmless without being evasive
- this session did not test whether AI labels agree with human ones
- the papers do, and we only quote them


# What we established, and what it sets up

- **Established**
    - RLAIF keeps RLHF's three stages and replaces the person by an AI labeler that reads a constitution
    - three cases: RLAIF by picking, direct RLAIF (the labeler's score is the reward), and Constitutional AI (rewrite for SFT, then pick)
    - AI labels move the cost from human time to the labeler's errors, biases and reading of the principles
- **Sets up**
    - the RL stage consumes any of these rewards, and a learned proxy can still be exploited
    - a source of grades that needs no labeler at all: a program that checks the answer (RLVR, RLLM-4)


## Why the paper wanted a constitution (aside, [Bai et al. 2022](https://arxiv.org/abs/2212.08073))

- **scalable supervision:** as models grow more capable, use them to help supervise other models
- **fewer human labels:** the only human input for harmlessness is the list of principles
